# Track 1 · 04 Diagnose — why is the head at chance?

The ridge fit came back healthy and empty:

```
lambda            7.296e-02  (eff.dof 510.0 of 512)
predictions vary  spread/|mean| = 0.1418
rank vs chance    0.4999 per-window, 0.4999 aggregated  (0.5 = chance)
top-5 vs chance   2.50% aggregated vs 2.50%
```

Nothing collapsed, the fit was barely regularised, and the answer is exactly chance.
Averaging five repeats per image moved it by nothing, which is telling: repeat averaging
amplifies any real signal, so there is none to amplify.

Three explanations are consistent with that, and they call for completely different work:

1. **The pipeline is broken** — windows paired with the wrong targets. Then no head can
   ever work and everything downstream is wasted effort.
2. **Mean-pooling destroyed it.** With one time patch, REVE's 32 tokens *are* the 32
   electrodes, so pooling averages occipital signal into 31 other channels.
3. **REVE's representation does not linearly carry image identity** on 1 s RSVP windows.

This notebook separates them, cheaply, with no training loop.

| test | what it rules out | cost |
|---|---|---|
| decode **subject** from the same features | a broken pipeline | seconds |
| ridge from **each electrode alone** | pooling as the culprit | ~30 s |
| ridge from the **best electrodes concatenated** | how much pooling costs | seconds |

The per-electrode test is the decisive one and it sidesteps the memory problem: fitting all
16,384 unpooled features at once needs a gigabyte-scale Gram matrix, whereas 32 fits of 512
features each answer the same question. If any single electrode ranks above chance, the
information survived the encoder and pooling is what lost it.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'
TOKEN_DIR = '/content/tokens'    # written by track1_02_features, ephemeral

SEED = 33
N_LAMBDA = 15
TOP_CHANNELS = 8                 # how many electrodes to concatenate in the last test

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
PREP = ROOT / 'Track1' / 'prepared'
FEAT = ROOT / 'Track1' / 'features'
TOKENS = Path(TOKEN_DIR)
rng = np.random.default_rng(SEED)


def l2(a, axis=-1):
    return a / (np.linalg.norm(a, axis=axis, keepdims=True) + 1e-12)


def rank_pct(pred, y_ids, emb):
    """Mean fraction of the gallery ranking below the true image. 0.5 is chance."""
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    tc = sims[np.arange(len(sims)), truth]
    return float((sims < tc[:, None]).mean())


def top5(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    k = min(5, len(gal))
    topk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, topk)]), 100.0 * k / len(gal)


def ridge_eval(Xa, Ya, Xb, yb, emb, lambdas):
    """Fit on (Xa, Ya), return the best rank on (Xb, yb) over the lambda grid."""
    mu, sd = Xa.mean(0), Xa.std(0) + 1e-8
    Xac = (Xa - mu) / sd
    ym = Ya.mean(0)
    d, V = np.linalg.eigh(Xac.T @ Xac)
    VtA = V.T @ (Xac.T @ (Ya - ym))
    Xbc = (Xb - mu) / sd
    out = []
    for lam in lambdas * max(d.mean(), 1e-12):
        pred = Xbc @ (V @ (VtA / (d + lam)[:, None])) + ym
        out.append((rank_pct(pred, yb, emb), lam, pred))
    return max(out, key=lambda t: t[0])


LAMBDAS = np.logspace(-4, 4, N_LAMBDA)
print(f'grid {N_LAMBDA} points, scaled to each fit')

In [ ]:
f = np.load(PREP / 'features_pooled.npz', allow_pickle=True)
t = np.load(PREP / 'image_embeddings.npz', allow_pickle=True)
emb = l2(t['emb'].astype('float32'))

Xtr, ytr, str_ = f['X_train'].astype('float64'), f['y_train'], f['subj_train']
Xte, yte, ste = f['X_test'].astype('float64'), f['y_test'], f['subj_test']
print(f'pooled  train {Xtr.shape}  test {Xte.shape}')
print(f'targets {emb.shape}, {len(np.unique(yte))} test images')

# Electrode names, so the per-channel result can be read anatomically rather than
# as a list of indices. A topographically coherent answer is itself evidence.
import mne
mne.set_log_level('ERROR')
edf = sorted((ROOT / 'Track1' / 'raw').rglob('*.edf'))[0]
CH = mne.io.read_raw_edf(edf, preload=False).ch_names

have_tokens = TOKENS.exists() and any(TOKENS.glob('*.npy'))
print(f'tokens  {"found" if have_tokens else "MISSING"} at {TOKENS}')
if not have_tokens:
    print('  /content is wiped between sessions. Re-run track1_02_features in this')
    print('  session to regenerate them (about 2 minutes), then re-run this notebook.')

## Test 1 — can these features decode *subject*?

Subject and session differences dominate EEG; they are the easiest thing in the data. If a
linear map cannot recover subject identity from the same pooled features, the problem is
upstream of the head and no amount of head work will help.

This is a genuine control, not a formality: it is trained and tested on different *runs*,
so it cannot win by memorising a recording.

In [ ]:
subs = sorted(set(str_.tolist()) | set(ste.tolist()))
s2i = {s: i for i, s in enumerate(subs)}
Ytr_s = np.eye(len(subs))[[s2i[s] for s in str_]]
true_te = np.array([s2i[s] for s in ste])

mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
Xa, Xb = (Xtr - mu) / sd, (Xte - mu) / sd
d, V = np.linalg.eigh(Xa.T @ Xa)
VtA = V.T @ (Xa.T @ (Ytr_s - Ytr_s.mean(0)))

rows = []
for lam in LAMBDAS * d.mean():
    pred = Xb @ (V @ (VtA / (d + lam)[:, None])) + Ytr_s.mean(0)
    hit = pred.argmax(1) == true_te
    bal = np.mean([hit[true_te == c].mean() for c in range(len(subs))])
    rows.append((bal, lam))
bal, lam_s = max(rows)
chance_s = 1.0 / len(subs)
print(f'subject decoding (train runs -> test runs, {len(subs)} subjects)')
print(f'  balanced accuracy {bal:.1%}   chance {chance_s:.1%}   lift {bal / chance_s:.1f}x')
if bal < chance_s + 0.5 * (1 - chance_s):
    print('\n  The features cannot even recover subject identity. Suspect the pipeline')
    print('  before the head: window/target alignment, epoching, or normalisation.')
else:
    print('\n  Features carry real structure, so the pipeline is alive and the image')
    print('  result is about the task, not about plumbing.')

## Test 2 — does any single electrode carry image information?

One ridge per electrode, 512 features each. Pooling averages these 32 tokens together, so if
any one of them ranks above chance on its own, the information existed and averaging is what
lost it.

In [ ]:
if have_tokens:
    import re

    def load_split(split):
        Xs, ys, ss = [], [], []
        for shard in sorted(FEAT.glob('*.npz')):
            d = np.load(shard, allow_pickle=True)
            if str(d['split']) != split:
                continue
            tok = TOKENS / f'{shard.stem}.npy'
            if not tok.exists():
                continue
            tk = np.load(tok).astype('float32')        # (n, chan, 512)
            keep = d['image_idx'] >= 0
            Xs.append(tk[keep]); ys.append(d['image_idx'][keep])
            ss.append(np.full(int(keep.sum()), str(d['subject'])))
        return np.concatenate(Xs), np.concatenate(ys), np.concatenate(ss)

    Ttr, ytr_t, _ = load_split('train')
    Tte, yte_t, _ = load_split('test')
    n_ch = Ttr.shape[1]
    print(f'tokens  train {Ttr.shape}  test {Tte.shape}\n')

    Ytr_e = emb[ytr_t]
    per_ch, ch_preds = [], []
    for c in range(n_ch):
        r, lam, pr = ridge_eval(Ttr[:, c].astype('float64'), Ytr_e,
                                Tte[:, c].astype('float64'), yte_t, emb, LAMBDAS)
        per_ch.append(r)
        ch_preds.append(pr)
    per_ch = np.array(per_ch)

    order = np.argsort(-per_ch)
    print('  electrode   rank_pct   (0.5 = chance)')
    for i in order[:10]:
        print(f'  {CH[i]:<10}  {per_ch[i]:.4f}')
    print('  ...')
    print(f'  {CH[order[-1]]:<10}  {per_ch[order[-1]]:.4f}   (worst)')
    print(f'\nbest {per_ch.max():.4f}   median {np.median(per_ch):.4f}   '
          f'pooled baseline 0.4999')
    if per_ch.max() > 0.51:
        occ = [CH[i] for i in order[:5]]
        print(f'\n  At least one electrode beats chance clearly. Pooling is destroying it.')
        print(f'  Top five: {", ".join(occ)}')
        print('  If those are occipital and parieto-occipital, the result is')
        print('  topographically coherent, which no plumbing bug would produce.')
    else:
        print('\n  No single electrode beats chance either. Pooling is NOT the bottleneck;')
        print('  the encoder output does not linearly carry image identity here.')
else:
    print('skipped: no token cache in this session')

## Test 2b — is that maximum beyond its own null?

Reporting the best of 32 fits is a selection, and the maximum of 32 noisy estimates sits
above chance even when nothing is there. So the null has to include the selection: permute
the test labels, recompute the *same* max-over-electrodes statistic, and see where the
observed value falls.

Without this the per-electrode result is not evidence, it is a leaderboard of noise.


In [ ]:
if have_tokens:
    N_PERM = 300
    gal = np.unique(yte_t)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in yte_t])
    Gm = l2(emb[gal])

    def ranks_of(pred, truth_pos):
        sims = l2(pred) @ Gm.T
        tc = sims[np.arange(len(sims)), truth_pos]
        return float((sims < tc[:, None]).mean())

    null_max = []
    for _ in range(N_PERM):
        perm = rng.permutation(truth)
        null_max.append(max(ranks_of(p, perm) for p in ch_preds))
    null_max = np.array(null_max)
    pval = float((null_max >= per_ch.max()).mean())

    print(f'observed max over {n_ch} electrodes   {per_ch.max():.4f}')
    print(f'null max ({N_PERM} label permutations) mean {null_max.mean():.4f}  '
          f'sd {null_max.std():.4f}  95th {np.percentile(null_max, 95):.4f}')
    print(f'p = {pval:.3f}')

    occ = [i for i, c in enumerate(CH) if c in ('O1', 'O2', 'Oz')]
    place = sorted(np.argsort(-per_ch).tolist().index(i) for i in occ)
    print(f'\npositions of O1/O2/Oz among {n_ch}: {place}  (0 = best)')

    print()
    if pval < 0.05:
        print('Signal is real and survives the selection. But read the effect size, not')
        print('the p-value: a significant shift in mean rank is not the same as a usable')
        print('top-5. Top-5 of 200 is the top 2.5% of the gallery, which needs the true')
        print('image concentrated near the very top, not merely shifted a little.')
    else:
        print('Not distinguishable from the null once selection is accounted for.')
        print('Do not build on this.')
else:
    print('skipped: no token cache in this session')


## Test 3 — the best electrodes together

If individual electrodes carry a little, concatenating the best of them should carry more.
This is a measurement, not a submission: selecting electrodes by name is not portable across
montages, so it tells us what pooling costs rather than what to ship.

In [ ]:
if have_tokens and per_ch.max() > 0.505:
    sel = order[:TOP_CHANNELS]
    Xa = Ttr[:, sel].reshape(len(Ttr), -1).astype('float64')
    Xb = Tte[:, sel].reshape(len(Tte), -1).astype('float64')
    r, lam, pred = ridge_eval(Xa, emb[ytr_t], Xb, yte_t, emb, LAMBDAS)
    acc, chance = top5(pred, yte_t, emb)
    print(f'{TOP_CHANNELS} best electrodes ({", ".join(CH[i] for i in sel)}), '
          f'{Xa.shape[1]} features')
    print(f'  rank   {r:.4f}   (pooled 0.4999, best single {per_ch.max():.4f})')
    print(f'  top-5  {acc:.2f}%  vs chance {chance:.2f}%')
elif have_tokens:
    print('skipped: no individual electrode cleared chance, so concatenating them')
    print('is not going to help. The bottleneck is upstream of the head.')
else:
    print('skipped: no token cache in this session')